# Modul 2: Big Data Ecosystem and Architectures
Notebook ini digunakan untuk melakukan uji coba integrasi lingkungan **Apache Spark (PySpark)** dengan menerapkan dataset stunting kesehatan anak, sesuai dengan panduan praktikum Modul 2.

In [1]:
!pip install pyspark

     -------------------------------------- 450.1/450.1 MB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 203.0/203.0 kB 2.5 MB/s eta 0:00:00
  Running setup.py install for pyspark: started
  Running setup.py install for pyspark: still running...
  Running setup.py install for pyspark: still running...
  Running setup.py install for pyspark: still running...
  Running setup.py install for pyspark: finished with status 'done'


  DEPRECATION: pyspark is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import pustaka SparkSession dari PySpark
from pyspark.sql import SparkSession

# Inisialisasi Spark Session untuk ekosistem pemrosesan terdistribusi
spark = SparkSession.builder \
    .appName("Modul2-StuntingEcosystem") \
    .getOrCreate()

print("Sesi Spark berhasil dimulai!")
print("Versi Spark yang digunakan:", spark.version)

Sesi Spark berhasil dimulai!
Versi Spark yang digunakan: 4.2.0


### Memuat Dataset Stunting (Sampel Bersih Modul 1)
Pada tahap ini, kita memuat data sampel bersih (`stunting_sample.csv`) yang telah dipersiapkan dari hasil eksplorasi Modul 1 ke dalam struktur DataFrame PySpark.

In [3]:
# Menentukan path file dataset sampel bersih dari Modul 1
file_path = "../Module 1/data/stunting_sample.csv" 

# Memuat CSV ke dalam PySpark DataFrame dengan otomatis mendeteksi skema
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)

# Menampilkan informasi dasar dataset
print(f"Total Baris dalam DataFrame Spark: {df_spark.count():,}")
print(f"Total Kolom: {len(df_spark.columns)}")

print("\n--- Skema Kolom Dataset ---")
df_spark.printSchema()

Total Baris dalam DataFrame Spark: 130,000
Total Kolom: 16

--- Skema Kolom Dataset ---
root
 |-- ID_Anak: string (nullable = true)
 |-- Provinsi: string (nullable = true)
 |-- Kabupaten_Kota: string (nullable = true)
 |-- Kecamatan: string (nullable = true)
 |-- Jenis_Kelamin: string (nullable = true)
 |-- Usia_Bulan: integer (nullable = true)
 |-- Berat_Lahir_kg: double (nullable = true)
 |-- Panjang_Lahir_cm: double (nullable = true)
 |-- Berat_Aktual_kg: double (nullable = true)
 |-- Tinggi_Aktual_cm: double (nullable = true)
 |-- Z_Score_TB_U: double (nullable = true)
 |-- Status_Stunting: string (nullable = true)
 |-- Riwayat_ASI_Eksklusif: string (nullable = true)
 |-- Akses_Sanitasi: string (nullable = true)
 |-- Sumber_Air_Minum: string (nullable = true)
 |-- Pendapatan_Keluarga: string (nullable = true)



### Inspeksi Data Teratas
Melihat beberapa baris pertama dari DataFrame Spark untuk memastikan format data sudah terbaca dengan benar.

In [4]:
# Menampilkan 5 baris teratas dataset stunting
print("--- 5 Baris Teratas Dataset Stunting (PySpark) ---")
df_spark.show(5, truncate=False)

--- 5 Baris Teratas Dataset Stunting (PySpark) ---
+-----------------+-------------------+---------------+-------------+-------------+----------+--------------+----------------+---------------+----------------+------------+---------------+---------------------+--------------+----------------------+-------------------+
|ID_Anak          |Provinsi           |Kabupaten_Kota |Kecamatan    |Jenis_Kelamin|Usia_Bulan|Berat_Lahir_kg|Panjang_Lahir_cm|Berat_Aktual_kg|Tinggi_Aktual_cm|Z_Score_TB_U|Status_Stunting|Riwayat_ASI_Eksklusif|Akses_Sanitasi|Sumber_Air_Minum      |Pendapatan_Keluarga|
+-----------------+-------------------+---------------+-------------+-------------+----------+--------------+----------------+---------------+----------------+------------+---------------+---------------------+--------------+----------------------+-------------------+
|ID_CHILD_00996419|Sumatera Barat     |Bukittinggi    |Kecamatan_9  |Perempuan    |51        |2.61          |51.69           |24.62          |

### Agregasi Data Berdasarkan Wilayah (Provinsi)
Melakukan pengelompokan (*grouping*) dan perhitungan rata-rata Z-Score menggunakan operasi terdistribusi Spark.

In [5]:
# Agregasi: Rata-rata Z-Score TB/U per Provinsi
if "Provinsi" in df_spark.columns and "Z_Score_TB_U" in df_spark.columns:
    summary_spark = df_spark.groupBy("Provinsi").avg("Z_Score_TB_U").withColumnRenamed("avg(Z_Score_TB_U)", "Rata_Z_Score")
    
    print("\n--- Hasil Agregasi: Rata-rata Z-Score per Provinsi ---")
    summary_spark.show()

# Menghentikan sesi Spark setelah selesai digunakan
spark.stop()
print("\nSesi Spark berhasil dihentikan.")


--- Hasil Agregasi: Rata-rata Z-Score per Provinsi ---
+-------------------+-------------------+
|           Provinsi|       Rata_Z_Score|
+-------------------+-------------------+
|     Sumatera Utara|-0.6267778205621869|
|     Sumatera Barat|-0.5896103896103895|
|   Sulawesi Selatan|-0.5964317553027974|
|         Jawa Barat|-0.6022826253642082|
|Nusa Tenggara Timur|-0.5928146278020042|
|            Lampung|-0.6221200779727099|
|   Sumatera Selatan|-0.6040210001510807|
|             Banten|-0.6008693291973628|
|         Jawa Timur|-0.6044409339600837|
|        Jawa Tengah| -0.592126123358169|
+-------------------+-------------------+


Sesi Spark berhasil dihentikan.
